# ETL Pipeline - Network Inventory

---

**⚠️ DATA PRIVACY NOTICE**

This notebook uses **sample data** for demonstration purposes. All data has been:
- Reduced in volume (representative subset)
- Anonymized to remove sensitive information
- Modified to protect business confidentiality

The code, methodology, and analysis logic reflect the actual production implementation, but numerical results and specific values are for illustration only.

---

## Overview

This ETL pipeline consolidates electrical network inventory data from multiple SAP exports into a single, analysis-ready dataset.

## Business Context
**Problem:** Orgainzing the inventory of medium voltage "technical units", cleaning and adding features of high importance for further analysis

**Impact:** This database is widely used because each analysis starts from a "technical unit" of the medium voltage network.

This notebook reads input files from `../data/input/` and generates output in `../data/output/`

**Input data sources:**
1. **MOD_ENTREGA - Delivery Modules** `SEG_MT_YYYYMM_MOD_ENTREGA_sample.csv`
2. **UT - Technical Units** `SEG_MT_YYYYMM_UT_TODAS_sample.csv`
3. **MPROTEC - Protection Modules** `MPROTEC_sample.xlsx`
4. **Jerarquia - Network Hierarchy** `Jerarquía_sample.xlsx`

**Output:**
- `inventory_sample.parquet` - Consolidated inventory with ~500K+ rows (one per network component). This sample has ~ 9700 rows.

**Key transformations:**
- Classify cable types (API vs SECO/XLPE) and joint types
- Assign a category for each Type component
- Assign protection module hierarchy
- Geocode components (latitude/longitude)
- Link components to their electrical segments (TPLMA/MM)

## Glossary

- MOD_ENTREGA - Delivery Modules: defines electrical segments and their sequence
- UT - Technical Units: all network components (cables, poles, joints, transformers)
- MPROTEC - Protection Modules: protective equipment assignments
- Jerarquia - Network Hierarchy: substations and feeders
- TPLMA: is the UT upstream of the UT, for some UTs is the MM or Delivery Module
- VANO: segment of aerial line
- SECCIONADOR: switcher installed in lines for operation
- TIPO_OBJ_TEC: main category of UT

In [1]:
# Imports
import os
import pandas as pd
import datetime
import numpy as np

# Direction of files
directorio = '../data/input'
output_dir = '../data/output'

## Functions

## Business Logic: Joint Classification

Electrical cable joints are critical maintenance points. We classify them by the cable types they connect:

- **SECO-SECO**: Connects two dry-insulated (XLPE) cables - Modern, lower risk
- **API-API**: Connects two paper-insulated cables - Older technology, higher risk
- **TRANSICION**: Connects API to SECO - Highest maintenance priority (material transition)
- **Sin secuencia**: Cannot determine (missing data or isolated joint)

**Why this matters:** Transition joints (API⟷SECO) have higher failure rates due to different insulation characteristics and require special maintenance attention.

The `tipo_de_empalme()` function implements this logic by looking at the sequence of components in each electrical segment.

In [89]:
## For each joint, I see the details of the previous and next cable in the sequence (SECO o API) and define the splice type (SECO-SECO, TRANSICION, API-API, Sin secuencia)

def tipo_de_empalme(row):
    if row['TIPO_OBJ_TEC'] != 'EMPALME MT':
        return 'NO'
    mm = row['TPLMA']
    secuencia_actual = row['SECUENCIA']
    
    # Filter the DataFrame 'cables' by TPLMA
    mismo_tplma = cables[cables['TPLMA'] == mm]

    # We look for the record with the closest backward sequence
    anterior_df = mismo_tplma[mismo_tplma['SECUENCIA'] < secuencia_actual]
    # We look for the record with the closest forkward sequence
    posterior_df = mismo_tplma[mismo_tplma['SECUENCIA'] > secuencia_actual]

    if anterior_df.empty or posterior_df.empty:
        return 'Sin secuencia'

    # .idxmax() and .idxmin() give us the index of the nearest value
    idx_anterior = anterior_df['SECUENCIA'].idxmax()
    idx_posterior = posterior_df['SECUENCIA'].idxmin()

    anterior_val = anterior_df.loc[idx_anterior, 'detalle']
    posterior_val = posterior_df.loc[idx_posterior, 'detalle']

    # Cannot determinate type of joint
    if pd.isna(anterior_val) or pd.isna(posterior_val):
        return 'Sin secuencia'

    # Classification logic
    if anterior_val == 'SECO' and posterior_val == 'SECO':
        return 'SECO-SECO'
    
    if anterior_val == 'API' and posterior_val == 'API':
        return 'API-API'

    if (anterior_val == 'API' and posterior_val == 'SECO') or \
       (anterior_val == 'SECO' and posterior_val == 'API'):
        return 'TRANSICION'

    return 'OTRO'

## Reading input files

In [ ]:
# Month of the file #
mes = datetime.datetime.now().month - 1
anio = datetime.datetime.now().year
if mes <= 0:
    mes = 12
    anio = anio - 1

print(mes, anio)

if mes <= 9:
    mes = '0' + str(mes)

mes = '03'          # Fixed month for this example
anio = '2026'       # Fixed year for this example

print(mes, anio)

# Delivery modules File  #
print('Reading Módulos de Entrega')
Entrega = pd.read_csv(f'{directorio}/SEG_MT_{anio}{mes}_MOD_ENTREGA_sample.csv', sep='\t', encoding='latin1')
Entrega = Entrega.drop_duplicates(subset=['UT_OBJECTNAMEID'])

# UTs of Medium Voltage #
print('Reading UTs MT')
uts = pd.read_csv(f'{directorio}/SEG_MT_{anio}{mes}_UT_TODAS_sample.csv', sep='\t', encoding='latin1')

# Protection Modules #
print('Reading Módulos de Protección')
mprotec = pd.read_excel(f'{directorio}/MPROTEC_sample.xlsx')

# Hierarchy - SE
print('Reading Jerarquía')
alimentadores  = pd.read_excel(directorio + '/Jerarquia_sample.xlsx')
alimentadores = alimentadores[['SE','Emplaz']]
alimentadores.drop_duplicates(inplace=True)
alimentadores = alimentadores.rename(columns={'Emplaz' : 'EMPLAZAMIENTO','SE':'NOMBRE'})
alimentadores = alimentadores[~alimentadores['EMPLAZAMIENTO'].str.contains('SE', na=False)]
alimentadores = alimentadores[~alimentadores['EMPLAZAMIENTO'].isna()]

4 2026
03 2026
Reading Módulos de Entrega
Reading UTs MT
Reading Módulos de Protección
Reading Jerarquía


## Editing UTs dataframe

In [91]:
uts2 = uts.copy()

In [92]:
uts2.head()

,#TPLNR,DESCRIPT,TIPO_OBJ_TEC,INVNR,BEGRU,F_ALTA_SAP,F_BAJA_SAP,PTA_SERVICIO,TPLMA,EMPLAZAMIENTO,...,COOR_X,COOR_Y,CTRO_COSTO,CALLE,ALTURA_DESDE,ALTURA_HASTA,ENTRE_CALLE_1,ENTRE_CALLE_2,GPS,IDENTIFICADOR
0,AK983,PIQUETE AK983,PIQUETE_MT,5579339,MT,22/03/2016,NaN,31/10/2022,MM102269053,99111A,...,5596314.693,6146008.986,4424.0,CNO A MORENO*,15701,15799,NaN,NaN,"-34.8275 , -58.9472",AK983
1,AF329-004,PIQUETE AF329,PIQUETE_MT,94103736,MT,14/04/2024,NaN,01/01/1998,MM96025328,99212A,...,5630845.780,6199138.375,4439.0,ARROYO TORO,NaN,NaN,ARROYO TORO,NaN,"-34.3448 , -58.578",AF329
2,AF238-004,PIQUETE AF238,PIQUETE_MT,94087427,MT,14/04/2024,NaN,01/01/1995,MM6326281,99212A,...,5630618.331,6194585.130,4439.0,ARROYO CURUBICA,NaN,NaN,RIO SARMIENTO,NaN,"-34.3859 , -58.5797",AF238
3,AG294-004,PIQUETE AG294,PIQUETE_MT,94178230,MT,22/03/2016,NaN,01/01/1998,MM95858046,99212A,...,5627541.249,6206856.433,4439.0,RIO PARANA DE LAS PALMAS,NaN,NaN,RIO PARANA DE LAS PALMAS,NaN,"-34.2757 , -58.615",AG294
4,AL287,PIQUETE AL287,PIQUETE_MT,5582662,MT,22/03/2016,NaN,01/01/1985,MM5582643,99111A,...,5597669.836,6148663.899,4424.0,NaN,NaN,NaN,NaN,NaN,"-34.8034 , -58.9327",AL287


### Component Type Classification

Network components need detailed classification for analysis:

**Cable & Line Types:**
- **API**: Paper-insulated cable (older, oil-filled)
- **SECO/XLPE**: Cross-linked polyethylene cable (modern, dry insulation)
- **Line configurations**: HORIZONTAL, VERTICAL, TRIANGULAR, LINE POST, COMPACTA (HORIZONTAL and VERTICAL are olders, LINE POST is new and others are special cases)

**Pole Types:**
- **PO/POD**: Madera (wooden poles)
- **PH**: Hormigón (concrete poles) - more endurable
- **PHME**: Metálico (metal poles)

**Transformation Center:**
- **CTA**: Platform-type center
- **CCE, CCS, CTN**: Various chamber center types

This classification enables:
1. Age analysis (API = older infrastructure)
2. Maintenance prioritization (wooden poles need more frequent inspection)
3. Equipment compatibility checking

In [93]:
uts2 = uts.copy()

# Fill LONGITUD of ALIM(feeder) with 0
uts2['LONGITUD'] = np.where(uts2['TIPO_OBJ_TEC'] == 'ALIM', 0, uts['LONGITUD'])

# Creo campo "detalle" en base a DESCRIPT con tipo de cable y tipo de línea ---------------------------------------------------

print('Completing detail of cable and line')
uts2['detalle'] = np.where(
    uts2['DESCRIPT'].str.endswith(' API'), 'API', # is API?

    np.where(
        uts2['DESCRIPT'].str.contains('XLPE'), 'SECO', # is seco (dry)?

            np.where(
                uts2['DESCRIPT'].str.startswith('VANO '), # is VANO?

                # Different line configuration of "VANO"
                np.where(
                    uts2['DESCRIPT'].str.endswith('T'), 'TRIANGULAR',

                    np.where(
                        uts2['DESCRIPT'].str.endswith('P. H'), 'HORIZONTAL PROT',

                        np.where(
                            uts2['DESCRIPT'].str.endswith('H'), 'HORIZONTAL DESN',

                            np.where(
                                uts2['DESCRIPT'].str.endswith('P. V'), 'VERTICAL PROT',

                                np.where(
                                    uts2['DESCRIPT'].str.endswith('V'), 'VERTICAL DESN',

                                    np.where(
                                        uts2['DESCRIPT'].str.endswith('P. LP'), 'LINE POST PROT',

                                        np.where(
                                            uts2['DESCRIPT'].str.endswith('LP'),
                                            'LINE POST DESN', 'COMPACTA'
                ))))))),
                #---------------------------------------------------------------------------------------------


                np.where(
                    uts2['DESCRIPT'].str.contains('TRAMO'),
                    np.where(uts2['DESCRIPT'].str.endswith('Al-I'), 'SECO', None), None))))

# Detail of poles --------------------------------------------------------------------------------------
# Assign 'Wood' to 'detail' for rows where 'OBJECT' begins with 'PO' or 'POD'
print('Completing detail of poles (Hormigon, Madera, Metálico)')
uts2['detalle'] = np.where((uts2['OBJETO'].str.startswith('PO') |
                            uts2['OBJETO'].str.startswith('POD')) &
                            (uts2['OBJETO'].notna()), 'Madera', uts2['detalle'])

# Assign 'Concrete' to 'detail' for rows where 'OBJECT' begins with 'PH'
uts2['detalle'] = np.where(uts2['OBJETO'].str.startswith('PH') &
                           (uts2['OBJETO'].notna()), 'Hormigon', uts2['detalle'])

# Assign 'Metallic' to 'detail' for rows where 'OBJECT' begins with 'PHME'
uts2['detalle'] = np.where(uts2['OBJETO'].str.startswith('PHME') &
                           (uts2['OBJETO'].notna()), 'Metalico', uts2['detalle'])

# We complete the details in CENTROS with the type of CENTRO --------------------------------------------------------
print('Completing detail of CENTRO')
lista = ['CCE', 'CCN',  'CCS', 'CCTN', 'CCTS',  'CTA2', 'CTAB', 'CTAMH',
'CTAMM', 'CTAS', 'CTE', 'CTI', 'CTN',  'CTS', 'CTSG']
otros = ['CCPE', 'CP', 'CTPE', 'CTPI', 'DM_CS', 'MCS3', 'MCS4', 'MCU3']
uts2['TIPO_OBJ_TEC'] = uts2['TIPO_OBJ_TEC'].str.replace('_', ' ')


for i in lista:      # Most common types of centers
    uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "CENTRO") &
                               (uts2['ALIAS'] == i) &
                               (uts2['OBJETO'].notna()), i, uts2['detalle'])

for o in otros:     # We group the less common types as OTRO
    uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "CENTRO") &
                               (uts2['ALIAS'] == o) &
                               (uts2['OBJETO'].notna()), 'OTRO', uts2['detalle'])

uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "CENTRO") &
                           (uts2['OBJETO'].isnull()), 'OTRO', uts2['detalle'])

# Detail of seccionadores (switcher) ----------------------------------------------------------------
print('Completing detail of switcher')
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'REC') & (uts2['OBJETO'].notna()),
                           'R', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECA') & (uts2['OBJETO'].notna()),
                           'K', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECBC') & (uts2['OBJETO'].notna()),
                           'BC', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECT') & (uts2['OBJETO'].notna()),
                           'S', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECU') & (uts2['OBJETO'].notna()),
                           'S', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECZ') & (uts2['OBJETO'].notna()),
                           'Z', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECZE') & (uts2['OBJETO'].notna()),
                           'Z', uts2['detalle'])



Completing detail of cable and line
Completing detail of poles (Hormigon, Madera, Metálico)
Completing detail of CENTRO
Completing detail of switcher


### Setting up TIPO_OBJ_TEC and Tipo_Cable

In [94]:
# Correcting TIPO_OBJ_TEC -----------------------------------------------------------------------
# Chamber type
print('Correcting TIPO_OBJ_TEC')
uts2['TIPO_OBJ_TEC'] = np.where((uts2['TIPO_OBJ_TEC'] == "CENTRO") &
                                (uts2['ALIAS'].isnull()), 'CAMARA', uts2['TIPO_OBJ_TEC'])

# Switcher type
uts2['TIPO_OBJ_TEC'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                                (uts2['ALIAS'].str.startswith('SE') |
                                 uts2['ALIAS'].str.startswith('REC')) &
                                (uts2['OBJETO'].notna()), 'SECCIONADOR', uts2['TIPO_OBJ_TEC'])


# Chamber type (other conditions)
uts2['TIPO_OBJ_TEC'] = np.where((uts2['TIPO_OBJ_TEC'] == "CENTRO") &
                                ((uts2['ALIAS'].str.startswith('C')) |
                                (uts2['ALIAS'].str.startswith('MC')) |
                                (uts2['ALIAS'].str.startswith('DM'))
                                ) &
                                (uts2['OBJETO'].notna()),
                                'CAMARA', uts2['TIPO_OBJ_TEC'])


# Platform type
uts2['TIPO_OBJ_TEC'] = np.where((uts2['TIPO_OBJ_TEC'] == "CAMARA") &
                                (uts2['ALIAS'].str.startswith('CTA')) &
                                (uts2['OBJETO'].notna()), 'PLATAFORMA', uts2['TIPO_OBJ_TEC'])


# Aux Inv 2--------------------------------
print('Calculating tipo_cable (type of cable)')
uts2['tipo_cable'] = np.where(uts2['DESCRIPT'].str.contains('TRAMO '), uts2['DESCRIPT'].str.replace('TRAMO ', ''),
                           np.where(uts2['DESCRIPT'].str.contains('VANO '), uts2['DESCRIPT'].str.replace('VANO ', ''), None))

Correcting TIPO_OBJ_TEC
Calculating tipo_cable (type of cable)


## Protection Module Hierarchy

The `MPROT` (Módulo de Protección) defines the electrical protection zone for each component.

**Hierarchy levels:**
- **Troncal** (Trunk): Main feeder line (starts with MP_R or just MP_[FEEDER])
- **Apendice Z**: Branch circuit off trunk (starts with MP_Z)
- **Apendice K**: Sub-branch off Z circuit (starts with MP_K)

**Why this matters:** 
- Protection devices are installed at specific points of the feeder
- At every fault you can know the protection that 'open' the circuit first
- By detecting this, smaller segments of the feeder can be prioritized for maintenance

**Voltage assignment:**
- Feeders on bars 1-6 (xx ≤ 70): 13 kV
- Feeders on bars 7-8 (xx > 70): 33 kV


In [95]:
# Completing MPROTs ---------------------------------------------------------------
print('Asigning MPROTs')

mprotecuts = mprotec.copy()
indices_to_drop = mprotec[(mprotec['PERUT'].str.startswith('MM')) &
                          (mprotec['PERUT'].notna())].index
mprotecuts = mprotec.drop(indices_to_drop)

uts2 = pd.merge(uts2, mprotecuts, how='left',  left_on='#TPLNR', right_on='PERUT')

mprotecmm = mprotec[(mprotec['PERUT'].str.startswith('MM')) & (mprotec['PERUT'].notna())]
uts2 = pd.merge(uts2, mprotecmm, how='left',  left_on='TPLMA', right_on='PERUT')

uts2['MPROT'] = uts2['MPROT_x'].combine_first(uts2['MPROT_y'])
uts2['NIVEL PROT'] = uts2['NIVEL PROT_x'].combine_first(uts2['NIVEL PROT_y'])
print(uts2[(uts2['NIVEL PROT'].notna()) & (uts2['NIVEL PROT']>0)]['NIVEL PROT'].unique())
uts2['NIVEL PROT'] = uts2['NIVEL PROT'].fillna(0)
uts2['NIVEL PROT'] = uts2['NIVEL PROT'].astype(int)
print(uts2[(uts2['NIVEL PROT'].notna()) & (uts2['NIVEL PROT']>0)]['NIVEL PROT'].unique())
print(uts2['NIVEL PROT'].isnull().sum())

# Hierarchy level
print('Asigning hierarchy level')
uts2['NIVEL_RED'] = np.where(uts2['MPROT'].str.startswith('MP_R'), 'Troncal',
                        np.where(uts2['MPROT'].str.startswith('MP_K'), 'Apendice K',
                            np.where(uts2['MPROT'].str.startswith('MP_Z'), 'Apendice Z', 'Troncal'
                            )
                        )
                    )

# MPROT of feeders ----------------------------------------------------------------
uts2['MPROT'] = uts2['MPROT'].astype(str)

uts2['MPROT'] = np.where(
    (uts2['MPROT'].str.endswith('A')) &
    (~uts2['MPROT'].str.contains('Z')) &
    (~uts2['MPROT'].str.contains('R')) &
    (~uts2['MPROT'].str.contains('K')),
    'MP_' + uts2['EMPLAZAMIENTO'],
    uts2['MPROT']
)

Asigning MPROTs
[4. 6. 7. 5. 8. 2. 3. 1.]
[4 6 7 5 8 2 3 1]
0
Asigning hierarchy level


## Other Fields

In [96]:
# Completing sequence -----------------------------------------------------------------------------------
print('Completing sequence')
uts3 = uts2.copy()

uts3 = pd.merge(uts3, Entrega[['UT_OBJECTNAMEID', 'ME_NOMBRE', 'ME_ORDEN', 'SECUENCIA', 'ME_PSEUDO']],
                how='left',  left_on='OBJECTNAMEID', right_on='UT_OBJECTNAMEID')

uts3['TPLMA'] = np.where((uts3['TPLMA'].str.startswith('MM')) & (uts3['TPLMA'] != uts3['ME_PSEUDO']), uts3['ME_PSEUDO'], uts3['TPLMA'])

uts3 = uts3.drop(['ME_PSEUDO'], axis=1)

uts3['ME_ORDEN'] = uts3['ME_ORDEN'].fillna(0).astype(int)
uts3['SECUENCIA'] = uts3['SECUENCIA'].fillna(0).astype(int)

# Completing detail in cells -------------------------------------------------------------------------------
print('Detail of cells')
uts3[['ELEMENTO',
      'TIPO ELEMENTO',
      'DESTINO',
      'TIPO DESTINO']] = uts3['IDENTIFICADOR'].str.split('-', expand=True)

mapeo_tipo_elemento = {      # Dictionary with cells type
    'CLMI': 'Medición Cliente MT',
    'FAEK': 'Seccionador Autodesc Fusible',
    'IMIMT': 'Interruptor',
    'RFMT': 'Ruptofusible',
    'SBCT': 'Seccionador',
    'SUMB': 'Seccionador'
}

mapeo_tipo_destino = {      # Dictionary with cells destination type
    0: '',
    '0': '',
    'CLMI': 'Cliente MT',
    'ECTA': 'Alim, Centro o EEMM',
    'FMIMT': 'Cliente MT',
    'TMMB': 'Transformador'
}
# Maping codes of cells
uts3['TIPO ELEMENTO'] = uts3['TIPO ELEMENTO'].fillna(0).replace(mapeo_tipo_elemento).astype(str)
uts3['TIPO DESTINO'] = uts3['TIPO DESTINO'].fillna(0).replace(mapeo_tipo_destino).astype(str)


# Separación Lat y Long ---------------------------------------------------------------------
print('Latitude and Longitude split')
uts3['Lat'] = uts3['GPS'].str.split(',').str.get(0)
uts3['Long'] = uts3['GPS'].str.split(',').str.get(1)

# Eliminando columnas -----------------------------------------------------------------------
print('Dropping unnecessary columns')
uts3 = uts3.drop(['PERUT_x', 'MPROT_x',
                  'NIVEL PROT_x', 'PERUT_y',
                  'MPROT_y', 'NIVEL PROT_y',
                  'COOR_X', 'COOR_Y', 'BEGRU',
                  'INVNR', 'DIVISION', 'OBJECTID',
                  'PARTIDO_GEO', 'LOCALIDAD_GEO',
                  'FLTYP', 'CTRO_COSTO', 'UT_OBJECTNAMEID',
                  'GPS','PERFIL','PARTE_OBJETO','SECCION'], axis=1)

# Changing specific cases of EMPLAZAMIENTO ----------------------------------------------------
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('6211-A', '6211A')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('6211-B', '6211B')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('6216-A', '6216A')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('6216-B', '6216B')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('16986 - FI', '16986A')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('16976 - FI', '16976A')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].fillna('00A')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].str.replace('_', '')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].str.replace('-', '')

# Voltage level to bars feeders 7 and 8 -----------------------------------------------
uts3['tipo_cable'] = np.where(uts3['TIPO_OBJ_TEC'] == 'ALIM',
                        np.where(
                            uts3['EMPLAZAMIENTO'].str[-3:].str[0:2].astype(int) > 70, '33', '13'),
                               uts3['tipo_cable'])

## Naming cells with CT name --------------------------------------------------------------------
print('Naming cells with CT name')
sec_mt = uts3[uts3['TIPO'] == 'SECC_MTCT'][['#TPLNR', 'TPLMA']]
sec_mt.rename(columns={'#TPLNR': 'trafo', 'TPLMA': 'celda'}, inplace=True)
uts3= pd.merge(uts3, sec_mt, how='left',  left_on='#TPLNR', right_on='celda')
uts3['tipo_cable'] = np.where(uts3['TIPO'] == 'CELDA', uts3['trafo'], uts3['tipo_cable'])


## Droping merge columns  --------------------------------------------------------------------
print('Droping merge columns')
uts3 = uts3.drop(['trafo','celda','ALIAS','SPRID'], axis=1)

## Filling ICC coordinates  -----------------------------------------------------------
print('Filling ICC coordinates')
ubis = uts3[uts3['TIPO_OBJ_TEC'] == 'CELDA MT'][['#TPLNR', 'Lat', 'Long']]
ubis.rename(columns={'#TPLNR': 'ubi', 'Lat': 'lati', 'Long': 'longi'}, inplace=True)
iccs = uts3[uts3['TIPO_OBJ_TEC'] == 'ICC'][['#TPLNR', 'TPLMA']]
iccs = pd.merge(iccs, ubis, how='left',  left_on='TPLMA', right_on='ubi')
iccs = iccs.drop(['ubi'], axis=1)
iccs.rename(columns={'#TPLNR': 'icc', 'TPLMA': 'secc'}, inplace=True)
uts3 = pd.merge(uts3, iccs, how='left',  left_on='#TPLNR', right_on='icc')
uts3['Lat'] = np.where(uts3['TIPO_OBJ_TEC'] == 'ICC', uts3['lati'], uts3['Lat'])
uts3['Long'] = np.where(uts3['TIPO_OBJ_TEC'] == 'ICC', uts3['longi'], uts3['Long'])
uts3 = uts3.drop(['lati'], axis=1)
uts3 = uts3.drop(['longi'], axis=1)
uts3 = uts3.drop(['secc'], axis=1)
uts3 = uts3.drop(['icc'], axis=1)

# Replacing 'FIN' of EMPLAZAMIENTO and ALIM columns -------------------------------------
uts3 = uts3[~((uts3['DESCRIPT'].str.contains('FIN')) & (uts3['TIPO_OBJ_TEC'] == 'ALIM'))]
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].str.replace('FIN ', '')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].str.replace('FIN', '')


# Adding Subestación (substation) --------------------------------------------------------------------------
uts3 = uts3.merge(alimentadores[['EMPLAZAMIENTO', 'NOMBRE']], how='left',  left_on='EMPLAZAMIENTO', right_on='EMPLAZAMIENTO')
uts3.rename(columns={'NOMBRE': 'SUBESTACION'}, inplace=True)

# Correcting particular date mistake PTO_SERVICIO -----------------------------------------------------------------
uts3['PTA_SERVICIO'] = np.where((uts3['PTA_SERVICIO'] == '23/10/5025') & (uts3['PTA_SERVICIO'].notna()), '23/10/2025', uts3['PTA_SERVICIO'])

Completing sequence
Detail of cells
Latitude and Longitude split
Dropping unnecessary columns
Naming cells with CT name
Droping merge columns
Filling ICC coordinates


## Calculating type of joints

In [97]:
# Creating cable_empalme df with sequence ------------------------------------------------------------
cables = uts3[
    (uts3['TIPO_OBJ_TEC'] == 'CABLE MT') 
    ][['#TPLNR', 'TIPO_OBJ_TEC', 'TPLMA', 'SECUENCIA', 'detalle']].reset_index(drop=True).copy()

empalmes = uts3[
    (uts3['TIPO_OBJ_TEC'] == 'EMPALME MT')
    ][['#TPLNR', 'TIPO_OBJ_TEC', 'TPLMA', 'SECUENCIA', 'detalle']].reset_index(drop=True).copy()

cable_empalme = pd.concat([cables, empalmes], axis=0).sort_values(['TPLMA', 'SECUENCIA']).reset_index()

In [98]:
# Applying function defining type of joint
cable_empalme['tipo_de_empalme'] = cable_empalme.apply(tipo_de_empalme, axis=1)

In [99]:
# Filling na and adding type of joint to uts3
cable_empalme['tipo_de_empalme'] = cable_empalme['tipo_de_empalme'].fillna('NO')
uts3 = uts3.merge(cable_empalme[['#TPLNR', 'tipo_de_empalme']], how='left', on='#TPLNR').copy()
uts3['detalle'] = uts3['detalle'].combine_first(uts3['tipo_de_empalme'])
uts3.drop('tipo_de_empalme', axis=1, inplace=True)

## Saving final inventory in parquet file

In [ ]:
uts3.to_parquet(output_dir + '/inventory_MT_sample.parquet', index=False)

In [101]:
uts3

,#TPLNR,DESCRIPT,TIPO_OBJ_TEC,F_ALTA_SAP,F_BAJA_SAP,PTA_SERVICIO,TPLMA,EMPLAZAMIENTO,AREA,OBJECTNAMEID,...,ME_NOMBRE,ME_ORDEN,SECUENCIA,ELEMENTO,TIPO ELEMENTO,DESTINO,TIPO DESTINO,Lat,Long,SUBESTACION
0,AK983,PIQUETE AK983,PIQUETE MT,22/03/2016,NaN,31/10/2022,MM102269053,99111A,Area1,5579339.0,...,MEL-12500-9|13057-9|B13-0254|K13.131MO|K13.76M...,23,29,AK983,0,None,,-34.8275,-58.9472,991-SE1
1,AF329-004,PIQUETE AF329,PIQUETE MT,14/04/2024,NaN,01/01/1998,MM96025328,99212A,Area2,94103736.0,...,MEL-40239-9|4562-9|4563-9|4705-9|K04-930|K62OL...,94,53,AF329,0,None,,-34.3448,-58.578,992-SE2
2,AF238-004,PIQUETE AF238,PIQUETE MT,14/04/2024,NaN,01/01/1995,MM6326281,99212A,Area2,94087427.0,...,MEL-40448-9|4359-1|K521OL|K56OL|S262OL|S263OL|,102,46,AF238,0,None,,-34.3859,-58.5797,992-SE2
3,AG294-004,PIQUETE AG294,PIQUETE MT,22/03/2016,NaN,01/01/1998,MM95858046,99212A,Area2,94178230.0,...,MEL-4657-1|4714-9|B04-1430|S658OL|,87,44,AG294,0,None,,-34.2757,-58.615,992-SE2
4,AL287,PIQUETE AL287,PIQUETE MT,22/03/2016,NaN,01/01/1985,MM5582643,99111A,Area1,5582662.0,...,MEL-12134-9|K12.109MO|,44,12,AL287,0,None,,-34.8034,-58.9327,991-SE1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9726,OA070,PIQUETE OA070,PIQUETE MT,22/03/2016,NaN,09/05/2002,MM6326545,99212A,Area2,6326363.0,...,MEL-4532-9|S04-1439|S699OL|Z698OL|,98,59,OA070,0,None,,-34.3705,-58.5839,992-SE2
9727,76052624,VANO 3x50/8 Al/Ac. P. LP,LINEA MT,02/10/2019,NaN,01/01/2019,MM87716688,99212A,Area2,76052624.0,...,MEC-4274-9|S04-419|S04-420|S04-786|S04-787|,109,62,NaN,0,NaN,,-34.4118,-58.5762,992-SE2
9728,76052326,VANO 3x50/8 Al/Ac. P. LP,LINEA MT,02/10/2019,NaN,01/01/2019,MM87716688,99212A,Area2,76052326.0,...,MEC-4274-9|S04-419|S04-420|S04-786|S04-787|,109,54,NaN,0,NaN,,-34.413,-58.5748,992-SE2
9729,97204721,VANO 3x50/8 Al/Ac. LP,LINEA MT,22/12/2024,NaN,19/12/2024,MM97171947,99111A,Area1,97204721.0,...,MEL-13227-9|13230-9|S12-0494|S13-0257|,26,99,NaN,0,NaN,,-34.7851,-58.9862,991-SE1


## ETL Results & Validation

### Final Dataset Statistics:

In [102]:
print("="*60)
print("INVENTORY ETL - FINAL RESULTS")
print("="*60)
print(f"\nTotal rows in final dataset: {len(uts3):,}")
print(f"\nComponent type breakdown:")
print(uts3['TIPO_OBJ_TEC'].value_counts())
print(f"\nCable type breakdown:")
print(uts3[uts3['TIPO_OBJ_TEC'].isin(['CABLE MT', 'LINEA MT'])]['detalle'].value_counts())
print(f"\nGeographic coverage:")
print(f"  - Unique feeders (EMPLAZAMIENTO): {uts3['EMPLAZAMIENTO'].nunique()}")
print(f"  - Unique substations: {uts3['SUBESTACION'].nunique()}")
print(f"  - Geocoded components: {uts3['Lat'].notna().sum():,} ({uts3['Lat'].notna().sum()/len(uts3)*100:.1f}%)")
print(f"\nOutput file: {output_dir}/inventory_sample.parquet")
print(f"File size: {os.path.getsize(output_dir + '/inventory_MT_sample.parquet') / 1024 / 1024:.1f} MB")
print("="*60)

# Show final structure
print("\nFinal dataset columns:")
print(uts3.columns.tolist())
print("\nSample rows:")
uts3.sample(3)


INVENTORY ETL - FINAL RESULTS

Total rows in final dataset: 9,731

Component type breakdown:
TIPO_OBJ_TEC
LINEA MT       4316
PIQUETE MT     3685
CELDA MT        631
TRAFO LOG       270
PLATAFORMA      256
SECCIONADOR     224
CABLE MT        135
MODULO           74
EMPALME MT       60
ICC              58
CAMARA           17
ALIM              5
Name: count, dtype: int64

Cable type breakdown:
detalle
VERTICAL DESN      966
LINE POST PROT     886
HORIZONTAL DESN    859
TRIANGULAR         828
LINE POST DESN     754
SECO               114
HORIZONTAL PROT     22
API                 21
COMPACTA             1
Name: count, dtype: int64

Geographic coverage:
  - Unique feeders (EMPLAZAMIENTO): 5
  - Unique substations: 5
  - Geocoded components: 9,608 (98.7%)

Output file: ../data/output/inventory_sample.parquet
File size: 0.6 MB

Final dataset columns:
['#TPLNR', 'DESCRIPT', 'TIPO_OBJ_TEC', 'F_ALTA_SAP', 'F_BAJA_SAP', 'PTA_SERVICIO', 'TPLMA', 'EMPLAZAMIENTO', 'AREA', 'OBJECTNAMEID', 'TIPO', 'P

,#TPLNR,DESCRIPT,TIPO_OBJ_TEC,F_ALTA_SAP,F_BAJA_SAP,PTA_SERVICIO,TPLMA,EMPLAZAMIENTO,AREA,OBJECTNAMEID,...,ME_NOMBRE,ME_ORDEN,SECUENCIA,ELEMENTO,TIPO ELEMENTO,DESTINO,TIPO DESTINO,Lat,Long,SUBESTACION
7269,5690284,VANO 3x50 Al. H,LINEA MT,13/04/2016,NaN,01/01/1971,MM87766313,99111A,Area1,5690284.0,...,MEL-13012-9|13024-9|K13-0238|K13.36MO|,11,27,NaN,0,NaN,,-34.9033,-58.9148,991-SE1
4592,6329828,VANO 3x50/8 Al/Ac. P. LP,LINEA MT,22/03/2016,NaN,01/01/2004,MM6329949,99212A,Area2,6329828.0,...,MEL-4288-9|S04-663|S311OL|,69,37,NaN,0,NaN,,-34.3982,-58.5954,992-SE2
6828,AA039-003,PIQUETE AA039,PIQUETE MT,22/03/2016,NaN,17/07/2007,MM91457323,99212A,Area2,19246286.0,...,MEL-4573-9|4574-9|4668-9|4701-9|S04-1363|S04-1...,92,61,AA039,0,None,,-34.3211,-58.597,992-SE2


### Data Quality Checks:
* All technical units assigned to electrical segments (TPLMA)
* Protection modules assigned to X% of components
* Geographic coordinates for Y% of field equipment
* Cable type classification complete for all cables/lines
* Joint classification complete for all joints

### This dataset is ready for: 
* Failure analysis (join with outage data) 
* Maintenance planning (join with condition assessments) 
* Network modeling (export to GIS or network analysis tools)
